In [1]:
import torch
import json
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
MAX_LENGTH = 512

MODEL_LIST = [
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen/Qwen1.5-1.8B",
    "microsoft/phi-2"
]

DATA_PATH = "outputs_20_per_prompt.jsonl"

In [2]:
def load_jsonl_flat(path):
    pairs = []
    with open(path, "r") as f:
        for line in f:
            item = json.loads(line)
            pairs.append((item["question"], item["output"]))
    return pairs

pairs = load_jsonl_flat(DATA_PATH)

print(f"Total pairs: {len(pairs)}")

Total pairs: 20000


In [ ]:
fisher_paths = []

for model_name in MODEL_LIST:
    path = compute_fisher_for_model(model_name)
    fisher_paths.append(path)


Processing: TinyLlama/TinyLlama-1.1B-Chat-v1.0


100%|██████████| 2500/2500 [09:41<00:00,  4.30it/s]


Saved: fisher_TinyLlama_TinyLlama-1.1B-Chat-v1.0.pt

Processing: Qwen/Qwen1.5-1.8B


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
100%|██████████| 2500/2500 [13:14<00:00,  3.15it/s]


Saved: fisher_Qwen_Qwen1.5-1.8B.pt

Processing: microsoft/phi-2


...

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

 89%|████████▉ | 2232/2500 [17:33<02:23,  1.87it/s]

In [ ]:
def compute_fisher_for_model(model_name):

    print(f"\nProcessing: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)

    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    fisher_diag = {
        name: torch.zeros_like(param, device=DEVICE, dtype=torch.float32)
        for name, param in model.named_parameters()
        if param.requires_grad
    }

    total_batches = 0
    total_tokens = 0

    for i in tqdm(range(0, len(pairs), BATCH_SIZE)):
        batch = pairs[i:i+BATCH_SIZE]

        texts = [p + "\n" + r for p, r in batch]

        enc = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )

        input_ids = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100  

        num_tokens = (labels != -100).sum().item()
        total_tokens += num_tokens

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss * num_tokens

        model.zero_grad(set_to_none=True)
        loss.backward()
        
        for name, param in model.named_parameters():
            if param.grad is not None:
                fisher_diag[name] += (param.grad.detach().float() ** 2)

        total_batches += 1

    eps = 1e-8
    for name in fisher_diag:
        fisher_diag[name] = fisher_diag[name] / total_tokens + eps

    fisher_diag = {k: v.cpu() for k, v in fisher_diag.items()}

    save_path = f"fisher_{model_name.replace('/', '_')}.pt"
    torch.save(fisher_diag, save_path)

    print(f"Saved: {save_path}")

    del model
    torch.cuda.empty_cache()

    return save_path

In [ ]:
fisher_paths = []

for model_name in MODEL_LIST:
    path = compute_fisher_for_model(model_name)
    fisher_paths.append(path)